# 项目 —— 航空 AI 助手

## 练习目标

把本周学到的 **Chat Completions、Gradio、Tools（函数调用）** 串起来，做一个航空公司客户支持助理：能聊天，也能查票价 / 改票价。

## 怎么跑

1. `.env` 里准备 `OPENAI_API_KEY`（或改用本地 Ollama，见下方注释）
2. 从上到下运行；遇到 `launch()` 会开 Gradio UI
3. 后半段会用 SQLite（`prices.db`）持久化票价


In [ ]:
# ========== 导入：环境变量 / JSON / OpenAI / Gradio ==========

# 导入 os：读环境变量里的 API Key
import os
# 导入 json：解析 tool call 里的 arguments JSON 字符串
import json
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：Chat Completions + tools
from openai import OpenAI
# 导入 gradio：ChatInterface 快速搭客服聊天 UI
import gradio as gr


In [ ]:
# ========== 初始化：加载密钥、选定 MODEL、创建客户端 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 从环境读取 OpenAI Key（只用于体检打印；OpenAI() 也会自动读）
openai_api_key = os.getenv('OPENAI_API_KEY')
# 有 Key：打印前 8 字符做存在性检查（完整 Key 不落盘）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    # 排错文案保持英文原样
    print("OpenAI API Key not set")
    
# 本练习默认云端小模型；model id 字符串禁止改写
MODEL = "gpt-4.1-mini"
# 默认 OpenAI 云端客户端
openai = OpenAI()

# 作为替代方案，如果您想使用 Ollama 而不是 OpenAI
# 检查 Ollama 是否在本地为您运行（请参阅 week1/day2 练习），然后取消注释接下来的 2 行
# 模型=“llama3.2”
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [ ]:
# ========== System Message：航空公司客服人设（发给模型，正文不翻译） ==========

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
# ========== 第一版 chat：无 Tools，纯多轮对话 + Gradio ==========

def chat(message, history):
    # Gradio messages 可能含多余字段；只保留 role/content 再交给 API
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史 + 当前 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 非流式一次拿完整回复
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    # 取出助手文本返回给 Gradio 气泡
    return response.choices[0].message.content

# type="messages"：history 为消息列表格式
gr.ChatInterface(fn=chat, type="messages").launch()


## Tools（工具 / 函数调用）

Tools 是前沿模型很强的能力：你写好一个函数，并把它的 **JSON Schema 描述** 交给模型；模型可以在回复里「提议」调用该函数。

注意：模型**不会**直接在你的机器上执行代码——它只返回 `tool_calls`；真正执行仍由你的 Python 完成，再把结果塞回 messages。


In [ ]:
# ========== 内存票价表 + get_ticket_price 工具函数 ==========

# 目的地（小写 key）-> 票价字符串；后面 get 用 .lower() 对齐
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    # 调试：确认 LLM 触发了工具而不是瞎编票价
    print(f"Tool called for city {destination_city}")
    # 查表；没有则 Unknown…（返回给模型的英文句子保持原样）
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
# ========== 手动试跑工具：不经过 LLM ==========

get_ticket_price("London")


In [ ]:
# ========== Tool Schema：描述 get_ticket_price 给模型看 ==========

# OpenAI tools 需要特定字典结构（name / description / parameters JSON Schema）
# There's a particular dictionary structure that's 必需 to describe our function:

get_price_function = {
    # 函数名：必须与后面 handle 时比对的名字一致
    "name": "get_ticket_price",
    # description：模型靠这段决定「何时该调」；保持英文原样
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        # 调用时必填字段
        "required": ["destination_city"],
        # 禁止额外未知参数
        "additionalProperties": False
    }
}


In [ ]:
# ========== [Raga] 按同样模式增加 set_ticket_price 的 Schema ==========

# Used two properties destination city and price, made both of them 必需

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a flight ticket to a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city for which the price will be set",
            },
            "price": {
                "type": "number",
                # 原作者 description 如此；不改字符串以免影响行为/对照
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}


In [ ]:
# ========== [Raga] get_all_ticket_prices：无参数也要带空 properties ==========

# 克劳德的指示：
# 遵循与笔记本中的 price_function 相同的模式
# 为其提供适当的名称和描述（例如，“获取所有可用目的地的价格”）
# 重要提示：由于没有参数，参数仍应具有类型："object" 和 properties:{}（空对象）
# Make 必需 an empty array []

get_all_ticket_prices_function = {
    "name": "get_all_ticket_prices",
    "description": "Get prices for all available destinationss.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False
    }
}


In [ ]:
# ========== 组装 tools 列表：每个元素 type=function + function schema ==========

tools = [{"type": "function", "function": get_price_function},{"type": "function", "function": set_price_function},{"type":"function","function":get_all_ticket_prices_function}]


In [ ]:
# ========== 查看 tools 结构（确认三份 schema 都挂上了） ==========

tools


## 让 OpenAI「使用」我们的工具

流程有点绕：我们并不是让模型直接执行函数，而是：

1. 请求时带上 `tools=...`
2. 若 `finish_reason == "tool_calls"`，说明模型希望你跑某个函数
3. 你本地执行后，把 `role: tool` 的结果追加进 messages，再问模型一次，得到自然语言答复


In [ ]:
# ========== 第二版 chat：单次 tool_calls 分支（先只示范一条路径） ==========

def chat(message, history):
    # 规范化 Gradio history
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 把 tools schema 交给模型，允许它返回 tool_calls
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 模型选择调用工具，而不是直接答完
    if response.choices[0].finish_reason=="tool_calls":
        # 含 tool_calls 的 assistant 消息必须原样追加，模型才认 tool 结果
        message = response.choices[0].message
        # 本地执行工具，得到 role=tool 的字典
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)

      # 对于消息中的消息：
      # 打印（消息）
        # 第二次调用：带着工具结果生成最终自然语言（本版不再传 tools）
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_call：只处理第一个 tool_call（get_ticket_price） ==========

# 我们必须编写该函数 handle_tool_call：

def handle_tool_call(message):
    # 本版简化：只取第一条 tool_call
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        # arguments 是 JSON 字符串，需 loads 成 dict
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        # 返回给 API 的 tool 消息：必须带上对应的 tool_call_id
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response


In [ ]:
# ========== 启动 Gradio：试用「带单工具处理」的 chat ==========

gr.ChatInterface(fn=chat, type="messages").launch()


## 改进方向

- 在**一次**模型响应里处理**多个** `tool_calls`
- 支持模型**连续多轮**要工具（while 循环）


In [ ]:
# ========== 第三版 chat：一次响应里的多个 tool_calls -> extend 多条 tool 消息 ==========

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        # handle_tool_calls（复数）：返回 list[tool message]
        responses = handle_tool_calls(message)
        messages.append(message)
        # extend：把多条 tool 结果依次追加
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_calls：遍历 message.tool_calls，逐个执行 get_ticket_price ==========

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# ========== 再开 UI：验证「一次多个 tool_calls」路径 ==========

gr.ChatInterface(fn=chat, type="messages").launch()


In [ ]:
# ========== 第四版 chat：while 循环，允许模型连续多轮要工具 ==========

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 只要还在要工具，就执行 -> 追加 -> 再问（并继续传 tools）
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # 注意：循环内仍传 tools，模型才能链式调用（如先改价再查询）
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content


In [ ]:
# ========== 导入 sqlite3：把票价从内存 dict 迁到本地数据库 ==========

import sqlite3


In [ ]:
# ========== 建库建表：prices(city PRIMARY KEY, price REAL) ==========

DB = "prices.db"

# with 连接：退出时自动关闭
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # IF NOT EXISTS：重复跑笔记本不会报错
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()


In [ ]:
# ========== 重写 get_ticket_price：从 SQLite 读价 ==========

def get_ticket_price(city):
    # flush=True：工具日志尽快打到笔记本输出
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询防注入；city 存小写
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        # 有行取 result[0]；否则英文提示保持原样
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"


In [ ]:
# ========== 试查 London（若尚未写入可能显示无数据） ==========

get_ticket_price("London")


In [ ]:
# ========== set_ticket_price：UPSERT 写入/更新票价 ==========

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # ON CONFLICT DO UPDATE：同一城市重复设置则覆盖
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()


In [ ]:
# ========== get_all_ticket_prices：列出全部城市与价格 ==========

def get_all_ticket_prices():
    print ("DATABASE TOOL CALLED: Get all Ticket Prices", flush=True)
    with sqlite3.connect(DB) as conn: 
        cursor = conn.cursor()
        cursor.execute ('SELECT city, price FROM prices')
        result = cursor.fetchall()
        # 格式化成 "London: $799" 这类片段再 join
        formatted = [f"{city.capitalize()}: ${price}" for city, price in result]
        return "Available ticket prices: " + ", ".join(formatted)


In [ ]:
# ========== 打印当前库里全部票价 ==========

print (get_all_ticket_prices())


In [ ]:
# ========== 种子数据：批量 set 几个城市的票价 ==========

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)


In [ ]:
# ========== 验证写入：查 Tokyo ==========

get_ticket_price("Tokyo")


In [ ]:
# ========== 启动 UI：此时工具已接 SQLite（handle 仍可能未覆盖 set/get_all） ==========

gr.ChatInterface(fn=chat, type="messages").launch()


## 练习

扩展 `handle_tool_calls`：支持 **设置票价**（以及本笔记本里的 get_all）。下面一格是作者 [Raga] 的完整分发实现。


In [ ]:
# ========== [Raga] 完整 handle_tool_calls：get / set / get_all 三分支 ==========

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        # 查价
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        # 改价：执行 set 后回一条确认文案给模型
        elif tool_call.function.name == "set_ticket_price": 
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get ('price')
            set_ticket_price(city,price)
            responses.append({
                "role": "tool",
                "content": f"Price set to {price} for city {city}",
                "tool_call_id": tool_call.id
            })
        # 列出全部票价（无参数）
        elif tool_call.function.name == "get_all_ticket_prices":  #Added get all ticket prices
            all_prices = get_all_ticket_prices()
            responses.append({
                "role": "tool",
                "content": all_prices,
                "tool_call_id": tool_call.id
            })
         



    return responses


In [ ]:
# ========== 最终 Gradio：三角色工具 + while 链式调用 + SQLite ==========

gr.ChatInterface(fn=chat, type="messages").launch()


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width:150px; height:150px; vertical-align:middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">业务应用</h2>
            <span style="color:#181;">希望这几乎不需要说明！你已经能让 LLM「采取行动」：航空助手不仅能回答问题，还能通过工具与票价数据（乃至未来的预订 API）交互。</span>
        </td>
    </tr>
</table>
